# Lab6.1 — Procesare de imagine software și hibridă

**Un singur experiment, două implementări:** Resize → conversie în tonuri de gri → Sobel.

- **Varianta A:** toate operațiile sunt executate pe procesorul ARM.
- **Varianta B:** Resize este executat în PL (folosind overlay-ul din Lab5), iar conversia în tonuri de gri și Sobel rămân pe ARM.

**Rezultate cerute:** imagini corecte, eroare numerică, timpi medii și percentila 95, FPS de procesare și o interpretare inginerească a diferențelor.

Comparația de performanță utilizează aceleași trei rezoluții pentru ambele trasee: **320×240, 256×192 și 160×120**. Pentru fiecare rezoluție, rezultatul acceleratorului trebuie validat înainte de interpretarea benchmark-ului.

## 1. Două trasee de procesare

**A — software:** DDR → Resize (ARM) → Grayscale (ARM) → Sobel (ARM).

**B — hibrid:** DDR → DMA → Resize (PL) → DMA → DDR → Grayscale (ARM) → Sobel (ARM).

**Observație:** configurația B nu este un pipeline hardware Resize → Sobel. Imaginea redimensionată revine în DDR și este prelucrată ulterior de ARM.

**Întrebare preliminară:** rezultatele celor două trasee trebuie să fie identice pixel cu pixel? Discutați efectele rotunjirii și interpolării asupra operatorului Sobel.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from lab6_common import (load_test_image, locate_resizer, ResizerHW,
                         process_cpu, process_hybrid, measure_callable, compare_images)

image = load_test_image()
TARGET = (320, 240)  # latime, inaltime
print("Imaginea sursa:", image.shape, "Rezolutia iesirii:", TARGET)
print("Versiunea OpenCV:", cv2.__version__)
plt.figure(figsize=(9, 5))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Imaginea de test"); plt.axis("off"); plt.show()

## 2. Validarea funcțională a celor două trasee

Programul utilizează perechea `resizer.bit` / `resizer.hwh` din Lab5. Dacă fișierele lipsesc, testul software rămâne utilizabil, dar măsurarea FPGA nu este inventată.

Comparați imaginile Sobel și diferența absolută. De ce o diferență de numai un nivel de intensitate după Resize poate produce diferențe mai mari după calculul gradientului?

**Dacă această celulă nu se finalizează:** mecanismul DMA poate aștepta un flux incomplet. Versiunea corectată a funcțiilor comune oprește așteptarea după 5 secunde și afișează o eroare. În acest caz, opriți experimentul și reporniți kernelul înaintea altei încercări.

In [ ]:
out_cpu = process_cpu(image, TARGET)
hw = None
try:
    bit = locate_resizer()
    hw = ResizerHW(bit)
    print("Overlay folosit:", bit)
except (ImportError, FileNotFoundError, AttributeError, RuntimeError) as exc:
    print("Varianta hibrida nu este disponibila:", exc)

# Prima executie reala prin DMA; la timeout reporniti kernelul.
out_hybrid = process_hybrid(image, TARGET, hw) if hw is not None else None
panels = [(out_cpu, "A / ARM")]
if out_hybrid is not None:
    metrics = compare_images(out_cpu, out_hybrid)
    diff = np.abs(out_cpu.astype(np.int16) - out_hybrid.astype(np.int16))
    print("Diferente numerice:", metrics)
    panels.extend([(out_hybrid, "B / hibrid"), (diff, "Eroare absoluta")])
fig, axes = plt.subplots(1, len(panels), figsize=(5*len(panels), 4))
for ax, (img, label) in zip(np.atleast_1d(axes), panels):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Benchmark — procesarea completă

Comparăm întregul lanț **Resize + Grayscale + Sobel**, nu doar timpul unui accelerator.

- Pentru ARM, timpul include toate operațiile OpenCV.
- Pentru varianta hibridă, timpul include copierea în bufferul DMA, transferul, Resize în PL, transferul de ieșire, conversia în tonuri de gri și Sobel pe ARM.
- Inițializarea overlay-ului și alocarea inițială a bufferelor nu intră în timpul repetat.

**Progresie experimentală:** după verificarea DMA la 320 × 240, comparăm CPU și hibrid la **aceleași trei rezoluții**. Funcția hibridă reconfigurează registrele și bufferele DMA când se schimbă dimensiunile imaginii. Dacă apare un timeout DMA la o altă rezoluție, notați dimensiunile, reporniți kernelul și nu continuați benchmark-ul fără diagnostic.

In [ ]:
# Aceleasi rezolutii pentru comparatie corecta ARM versus hibrid.
SIZES_CPU = [(320, 240), (256, 192), (160, 120)]
# Acum ca transferul DMA la 320x240 a fost validat, extindem testele.
SIZES_HW = SIZES_CPU
REPEATS = 15
rows = []

for size in SIZES_CPU:
    a = measure_callable(lambda: process_cpu(image, size), repeats=REPEATS)
    rows.append((size, "A / ARM", a))
    print("ARM:", size, "finalizat")

if hw is not None:
    for size in SIZES_HW:
        print("Test hibrid:", size)
        output_h = process_hybrid(image, size, hw)  # configurare + verificare
        output_a = process_cpu(image, size)
        print("Diferenta ARM/hibrid:", size,
              compare_images(output_a, output_h))
        b = measure_callable(lambda: process_hybrid(image, size, hw),
                             repeats=REPEATS)
        rows.append((size, "B / hibrid", b))
        print("Hibrid:", size, "finalizat")


# Rezultate sintetizate automat din masuratorile efectuate mai sus.
from IPython.display import HTML, display

def format_ro(value, decimals=3):
    return ("{:." + str(decimals) + "f}").format(value).replace(".", ",")

arm_results = {size: data for size, traseu, data in rows if traseu == "A / ARM"}
hybrid_results = {size: data for size, traseu, data in rows if traseu == "B / hibrid"}
ordered_sizes = list(reversed(SIZES_CPU))  # de la rezolutia mica la cea mare

display(HTML("<h3 style='margin:20px 0 12px'>1. Analiza performanțelor</h3>"))

head = (
    "<table style='border-collapse:collapse;width:100%;max-width:850px;"
    "font-family:Arial,sans-serif;font-size:14px'>"
    "<thead><tr style='text-align:left;border-bottom:2px solid #d7dce3'>"
    "<th style='padding:9px'>Rezoluție</th>"
    "<th style='padding:9px;text-align:right'>ARM (ms)</th>"
    "<th style='padding:9px;text-align:right'>Hibrid (ms)</th>"
    "<th style='padding:9px;text-align:right'>Speed-up ARM/Hibrid</th>"
    "<th style='padding:9px'>Verdict</th>"
    "</tr></thead><tbody>"
)
lines = []
for size in SIZES_CPU:
    a = arm_results[size]["mean_ms"]
    h = hybrid_results.get(size)
    if h is None:
        cells = [format_ro(a), "—", "—"]
        verdict = "Hibrid nemăsurat"
    else:
        b = h["mean_ms"]
        speedup = a / b
        cells = [format_ro(a), format_ro(b), format_ro(speedup) + "×"]
        verdict = ("Aproape egal" if 0.95 <= speedup <= 1.05 else
                   "Hibrid mai rapid" if speedup > 1.05 else "ARM mai rapid")
    lines.append(
        "<tr style='border-bottom:1px solid #e5e7eb'>"
        "<td style='padding:9px'>" + str(size[0]) + "×" + str(size[1]) + "</td>"
        + "".join("<td style='padding:9px;text-align:right'>" + x + "</td>"
                  for x in cells)
        + "<td style='padding:9px'>" + verdict + "</td></tr>"
    )
display(HTML(head + "".join(lines) + "</tbody></table>"))

# Graficul afiseaza timpii reali, fara valori introduse manual.
x = np.arange(len(ordered_sizes))
labels = ["{}×{}".format(w, h) for w, h in ordered_sizes]
fig, ax = plt.subplots(figsize=(10.5, 5))
ax.plot(x, [arm_results[s]["mean_ms"] for s in ordered_sizes],
        marker="o", linewidth=2.0, color="#3B82F6", label="ARM")
if hybrid_results:
    x_h = [i for i, s in enumerate(ordered_sizes) if s in hybrid_results]
    y_h = [hybrid_results[ordered_sizes[i]]["mean_ms"] for i in x_h]
    ax.plot(x_h, y_h, marker="o", linewidth=2.0, color="#59BD7B",
            label="Hibrid FPGA + ARM")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(bottom=0)
ax.set_ylabel("Timp mediu (ms)")
ax.set_title("Latența procesării în funcție de rezoluție",
             loc="left", fontsize=14, pad=28, fontweight="bold")
ax.text(0, 1.015, "Resize → Grayscale → Sobel, fără achiziție video",
        transform=ax.transAxes, fontsize=10, color="#64748b")
ax.grid(axis="y", alpha=0.22, linestyle="--")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12),
          ncol=2, frameon=False)
fig.subplots_adjust(bottom=0.23, top=0.80, left=0.11, right=0.96)
plt.show()

if hybrid_results:
    display(HTML(
        "<h3 style='margin-top:16px'>Concluzie</h3>"
        "<p>Am demonstrat că avantajul accelerării Resize depinde de volumul "
        "operațiilor software eliminate în raport cu overhead-ul introdus "
        "de DMA și de transferurile datelor.</p>"
    ))
else:
    display(HTML("<p><b>Concluzie:</b> comparația cu FPGA nu poate fi "
                 "formulată fără măsurători hibride valide.</p>"))


## 4. Investigație: influența volumului de date

Alegeți o rezoluție suplimentară, formulați o ipoteză asupra performanței și verificați-o experimental.

Benchmark-ul anterior utilizează acum trei rezoluții identice pe ARM și pe traseul hibrid. Pentru o **a patra rezoluție**, notați mai întâi ipoteza și activați separat testul FPGA numai după verificarea celor trei configurații anterioare.

Distingeți clar:

- FPS de procesare (fără captură), calculat aici;
- FPS al aplicației complete, incluzând achiziția video (Lab6.3);
- throughput-ul unui pipeline hardware streaming, care necesită un alt overlay (Lab6.2).

In [ ]:
EXTRA_SIZE = (480, 320)  # modificati dupa formularea ipotezei
a = measure_callable(lambda: process_cpu(image, EXTRA_SIZE), repeats=20)
print("ARM, noua rezolutie:", a)

TEST_REZOLUTII_HARDWARE_SUPLIMENTARE = False  # True numai dupa verificarea DMA
if TEST_REZOLUTII_HARDWARE_SUPLIMENTARE and hw is not None:
    print("Test hardware suplimentar:", EXTRA_SIZE)
    _ = process_hybrid(image, EXTRA_SIZE, hw)
    b = measure_callable(lambda: process_hybrid(image, EXTRA_SIZE, hw),
                         repeats=20)
    print("Hibrid:", b)
    print("Raport ARM/hibrid:", a["mean_ms"] / b["mean_ms"])
else:
    print("Testul la noua rezolutie a fost efectuat doar pe ARM.")

## 5. Analiză și eliberarea resurselor

În raport răspundeți argumentat:

1. Unde este scrisă în DDR imaginea redimensionată în varianta B?
2. Reducerea numărului de operații ARM înseamnă automat mai puține transferuri de memorie?
3. De ce FPS-ul de procesare nu este echivalent cu FPS-ul unei camere USB?
4. Ce trebuie schimbat pentru a transmite pixelii direct de la Resize la Sobel în PL?

Includeți tabelul măsurătorilor, comparația imaginilor, ipoteza și concluziile. Eliberați bufferele înainte de a încărca alt overlay.

In [ ]:
if hw is not None:
    hw.close()
    hw = None
print("Lab6.1 finalizat. Bufferele DMA au fost eliberate.")